In [2]:
from pathlib import Path

import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---------- data path: ../data first, else search up the folder tree ----------
FILENAME = "plot5_g7_bond-yields.csv"
PATH = Path("../data") / FILENAME
if not PATH.exists():
    PATH = next(
        p for parent in [Path.cwd(), *list(Path.cwd().parents)[:4]]
        for p in parent.rglob(FILENAME)
    )
print("Using:", PATH.resolve())

# ---------- settings ----------
COUNTRY = "Germany"
START = None            # e.g. 2000 to zoom in like the FRED chart; None = whole series
GREY = "#676A86"

# ---------- data ----------
df = pd.read_csv(PATH, encoding="utf-8-sig", parse_dates=["date"])
df = df[["date", COUNTRY]].rename(columns={COUNTRY: "yield"})
df["yield"] = pd.to_numeric(df["yield"], errors="coerce")
df = df.dropna().sort_values("date")
if START:
    df = df[df["date"].dt.year >= START]
df["series"] = "10-year yield"      # constant field so the theme assigns the colour

first, last = df["date"].min(), df["date"].max()
print(f"{len(df)} monthly observations, {first:%b %Y} to {last:%b %Y}; "
      f"latest = {df['yield'].iloc[-1]:.2f}%")

# ---------- chart ----------
x = alt.X(
    "date:T", title=None,
    axis=alt.Axis(format="%Y", tickCount=alt.TimeIntervalStep(interval="year", step=10)),
)
y = alt.Y(
    "yield:Q", title="10-year government bond yield",
    axis=alt.Axis(labelExpr="datum.label + '%'"),
)

line = alt.Chart(df).mark_line(strokeWidth=1.8).encode(
    x=x, y=y, color=alt.Color("series:N", legend=None),
)
zero = alt.Chart(pd.DataFrame({"yield": [0]})).mark_rule(
    color=GREY, strokeWidth=0.8).encode(y=y)      # yields went negative, so mark zero

caption = alt.Title(
    text="Source: OECD via FRED",
    subtitle=[f"Germany: 10-year government bond yield, %, monthly, {first.year}–{last.year}."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color=GREY, subtitleColor=GREY, dy=12)

chart = (
    (zero + line)
    .properties(width=400, height=300)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor=GREY, titleColor=GREY))

styles.save(chart, path=".", name="germany_bund_yield", svg=True)
chart.save("germany_bund_yield.png", scale_factor=2.0)
chart

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Peterson Blogs/Roger Schneider/data/plot5_g7_bond-yields.csv
844 monthly observations, May 1956 to Aug 2026; latest = 3.18%


alt.LayerChart(...)